# Наочний приклад: API та REST — споживання й створення власного 🌐

**API** (Application Programming Interface) — це «контракт», за яким одна програма звертається до іншої. Найпоширеніший вид веб-API сьогодні — **REST**: спілкування відбувається через звичайні HTTP-запити, а дані передаються здебільшого у форматі **JSON**.

Навіщо це AI-інженеру:

- 🤖 **Виклик LLM та сервісів** — OpenAI, Anthropic тощо — це REST API.
- 🔌 **Інтеграції** — погода, курси валют, бази даних, пошук — усе через API.
- 🛠️ **Інструменти (tools) для агентів** — часто це обгортка над чужим API.
- 🚀 **Власний бекенд** — ви віддаєте свою AI-модель іншим через свій API.

Що пройдемо крок за кроком:

1. Як влаштований HTTP-запит: метод, URL, заголовки, тіло, статус-код
2. Перший `GET`-запит бібліотекою `requests`
3. Параметри запиту (query params) і заголовки (headers)
4. `POST` / `PUT` / `PATCH` / `DELETE` — повний CRUD
5. Статус-коди та обробка помилок (timeouts, `raise_for_status`)
6. Автентифікація: API-ключі та Bearer-токени
7. ⚡ Асинхронні запити через `httpx` (бонус)
8. 🏗️ Створюємо **власний REST API** на FastAPI
9. Валідація тіла через Pydantic + повний CRUD
10. Тестуємо свій API всередині ноутбука (`TestClient`)
11. Як запустити сервер (`uvicorn`) і відкрити інтерактивну документацію

> 💡 Розділи 1–7 працюють одразу (потрібен інтернет). Для розділів 8–11 встановіть FastAPI — є клітинка з командою.


---
## 1. Як влаштований HTTP-запит 📨

Будь-який REST-виклик — це **HTTP-запит**. У нього є:

| Частина | Що це | Приклад |
|---|---|---|
| **Метод** | що робимо з ресурсом | `GET`, `POST`, `PUT`, `PATCH`, `DELETE` |
| **URL** | адреса ресурсу (endpoint) | `https://api.site.com/users/7` |
| **Заголовки** | метадані запиту | `Authorization`, `Content-Type` |
| **Тіло (body)** | дані, які надсилаємо | JSON `{"name": "Іван"}` |

**Семантика методів (CRUD):**

| Метод | Дія | CRUD |
|---|---|---|
| `GET` | прочитати дані | **R**ead |
| `POST` | створити новий ресурс | **C**reate |
| `PUT` / `PATCH` | оновити (повністю / частково) | **U**pdate |
| `DELETE` | видалити | **D**elete |

У відповідь сервер повертає **статус-код** і, зазвичай, **тіло у JSON**.

**Класи статус-кодів:**

- `2xx` — успіх (`200 OK`, `201 Created`, `204 No Content`)
- `3xx` — перенаправлення
- `4xx` — помилка клієнта (`400 Bad Request`, `401 Unauthorized`, `404 Not Found`)
- `5xx` — помилка сервера (`500 Internal Server Error`)


---
## 2. Встановлення й перший запит 🛠️

Для **споживання** API використаємо бібліотеку `requests` (стандарт де-факто). Якщо її немає — розкоментуйте рядок встановлення.

Як «навчальний» сервер візьмемо **JSONPlaceholder** — безкоштовний фейковий REST API (`https://jsonplaceholder.typicode.com`). Він приймає всі методи й повертає правдоподібні дані, нічого насправді не змінюючи.

In [ ]:
# %pip install requests

import requests
print("requests версія:", requests.__version__)

BASE = "https://jsonplaceholder.typicode.com"

### Перший `GET`-запит

Запитаємо допис (post) з `id = 1`. Дивимось на **статус-код** і на **тіло**, перетворене з JSON у Python-словник методом `.json()`.

In [ ]:
resp = requests.get(f"{BASE}/posts/1", timeout=10)

print("Статус-код:", resp.status_code)        # 200 = OK
print("Content-Type:", resp.headers["Content-Type"])
print("Чи успішно (2xx)?", resp.ok)

data = resp.json()                              # JSON -> dict
print("\nТіло відповіді (dict):")
print(data)
print("\nЗаголовок допису:", data["title"])

---
## 3. Параметри запиту та заголовки 🎚️

Часто треба **відфільтрувати** або **сторінкувати** дані. Для цього в URL додають *query-параметри*: `...?userId=1`. У `requests` їх зручно передавати словником у `params=` — бібліотека сама зробить правильний URL (включно з екрануванням).

`headers=` дозволяє передати метадані: формат, мову, токен тощо.

In [ ]:
# Усі дописи користувача userId=1
resp = requests.get(
    f"{BASE}/posts",
    params={"userId": 1},
    headers={"Accept": "application/json"},
    timeout=10,
)

print("Підсумковий URL:", resp.url)            # бачимо ?userId=1
posts = resp.json()
print("Скільки дописів:", len(posts))
print("Перші три заголовки:")
for p in posts[:3]:
    print(" -", p["id"], p["title"][:40])

---
## 4. POST / PUT / PATCH / DELETE — повний CRUD ✍️

Тепер не лише читаємо, а й **змінюємо** дані. JSON у тілі найзручніше передавати через `json=` — `requests` сам серіалізує словник і проставить заголовок `Content-Type: application/json`.

> Примітка: JSONPlaceholder імітує зміни (повертає коректну відповідь зі статусом), але реально нічого не зберігає.

In [ ]:
# CREATE -> POST: створюємо новий допис
new_post = {"title": "Мій допис", "body": "Текст допису", "userId": 1}
r = requests.post(f"{BASE}/posts", json=new_post, timeout=10)
print("POST  ->", r.status_code, "(201 = Created)")
print("Сервер повернув створений ресурс із новим id:", r.json())

In [ ]:
created_id = 1  # умовно працюємо з ресурсом id=1

# UPDATE -> PUT: повна заміна ресурсу
r = requests.put(
    f"{BASE}/posts/{created_id}",
    json={"id": created_id, "title": "Новий заголовок", "body": "...", "userId": 1},
    timeout=10,
)
print("PUT   ->", r.status_code, "|", r.json()["title"])

# UPDATE -> PATCH: часткове оновлення (лише одне поле)
r = requests.patch(f"{BASE}/posts/{created_id}", json={"title": "Лише заголовок"}, timeout=10)
print("PATCH ->", r.status_code, "|", r.json()["title"])

# DELETE: видалення
r = requests.delete(f"{BASE}/posts/{created_id}", timeout=10)
print("DELETE->", r.status_code, "(200/204 = видалено)")

---
## 5. Статус-коди та обробка помилок ⚠️

Реальна мережа ненадійна: сервер може відповісти помилкою, а з'єднання — зависнути. Тому **завжди**:

- ставте `timeout=`, щоб запит не висів вічно;
- перевіряйте результат через `resp.raise_for_status()` — він кине виняток на `4xx`/`5xx`;
- ловіть `requests.exceptions.RequestException` (батьківський клас усіх мережевих помилок).

In [ ]:
# Запитуємо ресурс, якого немає -> сервер відповість 404
resp = requests.get(f"{BASE}/posts/999999", timeout=10)
print("Статус:", resp.status_code)

try:
    resp.raise_for_status()           # перетворює 4xx/5xx на виняток
    print("Дані:", resp.json())
except requests.exceptions.HTTPError as e:
    print("Спіймали HTTP-помилку:", e)

In [ ]:
# Узагальнена «надійна» обгортка над GET-запитом
def safe_get(url, **kwargs):
    try:
        r = requests.get(url, timeout=5, **kwargs)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("⏱️  Перевищено час очікування")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP {e.response.status_code}: сервер відповів помилкою")
    except requests.exceptions.RequestException as e:
        print(f"🔌 Мережева помилка: {e}")
    return None

print(safe_get(f"{BASE}/users/1"))      # успіх -> dict
print(safe_get(f"{BASE}/users/999999")) # 404 -> None

---
## 6. Автентифікація: API-ключі та Bearer-токени 🔑

Захищені API вимагають довести, хто ви. Найпоширеніші способи:

- **API-ключ у заголовку**: `headers={"X-API-Key": "..."}` або `Authorization: Bearer <token>`;
- **API-ключ у query-параметрі**: `params={"api_key": "..."}` (менш безпечно — світиться в URL).

Саме так ви звертаєтесь і до LLM-провайдерів. Наприклад, виклик OpenAI/Anthropic — це POST із заголовком `Authorization: Bearer $API_KEY` і JSON-тілом.

> 🔒 **Ніколи не пишіть ключ прямо в коді!** Тримайте його у змінних оточення (`os.environ`) або у `.env`.

In [ ]:
import os

# Ключ беремо з оточення; якщо його немає — підставляємо заглушку для демо
API_KEY = os.environ.get("DEMO_API_KEY", "demo-key-123")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# httpbingo.org віддзеркалює запит — зручно ПОБАЧИТИ, що саме ми надіслали
r = requests.get("https://httpbingo.org/bearer", headers=headers, timeout=10)
print("Статус:", r.status_code)
print("Сервер побачив наш токен:", r.json())

In [ ]:
# Схема типового виклику LLM-провайдера (псевдокод — без реального ключа НЕ виконуйте):
#
# resp = requests.post(
#     "https://api.openai.com/v1/chat/completions",
#     headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
#     json={
#         "model": "gpt-4o-mini",
#         "messages": [{"role": "user", "content": "Привіт!"}],
#     },
#     timeout=60,
# )
# print(resp.json()["choices"][0]["message"]["content"])

print("☝️ Будь-який виклик LLM — це звичайний POST-запит до REST API з токеном у заголовку.")

---
## 7. ⚡ Бонус: асинхронні запити через `httpx`

Коли треба зробити **багато запитів** (наприклад, опитати 100 endpoint-ів), послідовний `requests` повільний — кожен чекає на попередній. Бібліотека **`httpx`** має такий самий API, але вміє працювати **асинхронно**: запити летять «паралельно».

`httpx` також підтримує синхронний режим (`httpx.get(...)`) — це майже drop-in заміна `requests`.

In [ ]:
import httpx, asyncio, time

async def fetch(client, post_id):
    r = await client.get(f"{BASE}/posts/{post_id}", timeout=10)
    return r.json()["title"][:30]

async def fetch_many(ids):
    async with httpx.AsyncClient() as client:
        tasks = [fetch(client, i) for i in ids]
        return await asyncio.gather(*tasks)   # усі запити одночасно

start = time.perf_counter()
titles = await fetch_many(range(1, 11))       # у Jupyter можна await на верхньому рівні
print(f"Отримали {len(titles)} відповідей за {time.perf_counter() - start:.2f} c")
for t in titles[:5]:
    print(" -", t)

---
## 8. 🏗️ Створюємо власний REST API на FastAPI

Досі ми були **клієнтом**. Тепер станьмо **сервером**. **FastAPI** — сучасний фреймворк для Python: швидкий, з автоматичною валідацією через **Pydantic** (який ви вже знаєте!) та згенерованою документацією.

Встановіть FastAPI і тестовий клієнт (рядок нижче розкоментуйте за потреби):

In [1]:
# %pip install "fastapi[standard]" uvicorn httpx

import fastapi, pydantic
print("FastAPI версія:", fastapi.__version__)
print("Pydantic версія:", pydantic.VERSION)

FastAPI версія: 0.138.0
Pydantic версія: 2.13.4


### Мінімальний застосунок

Створюємо об'єкт `FastAPI()` і описуємо **маршрути (routes)** декораторами `@app.get`, `@app.post` тощо. Функція-обробник повертає звичайний `dict` або Pydantic-модель — FastAPI сам перетворить це на JSON-відповідь.

In [2]:
from fastapi import FastAPI

app = FastAPI(title="Мій перший API")

@app.get("/")
def root():
    return {"message": "Привіт, API!"}

@app.get("/ping")
def ping():
    return {"status": "ok"}

print("Застосунок створено. Маршрути:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(" ", sorted(route.methods - {"HEAD", "OPTIONS"}), route.path)

Застосунок створено. Маршрути:
  ['GET'] /openapi.json
  ['GET'] /docs
  ['GET'] /docs/oauth2-redirect
  ['GET'] /redoc
  ['GET'] /
  ['GET'] /ping


---
## 9. Параметри шляху, query та тіло з валідацією 🧩

FastAPI читає параметри прямо з анотацій типів функції:

- **Path-параметр** — частина URL: `/items/{item_id}` → аргумент `item_id: int`;
- **Query-параметр** — звичайний аргумент зі значенням за замовчуванням: `q: str = None`;
- **Тіло запиту** — аргумент типу **Pydantic-моделі**: FastAPI його **провалідує** і поверне зрозумілу 422-помилку, якщо дані погані.

In [ ]:
from pydantic import BaseModel, Field

# Pydantic-модель = контракт на тіло запиту (валідація безкоштовно)
class Item(BaseModel):
    name: str = Field(min_length=1, description="Назва товару")
    price: float = Field(gt=0, description="Ціна, > 0")
    in_stock: bool = True

@app.get("/items/{item_id}")          # item_id — path-параметр
def read_item(item_id: int, q: str | None = None):  # q — query-параметр
    result = {"item_id": item_id}
    if q:
        result["q"] = q
    return result

@app.post("/items", status_code=201)   # 201 Created
def create_item(item: Item):           # тіло -> Pydantic-модель -> валідація
    return {"created": item.model_dump(), "total_price": item.price}

print("Маршрути /items додано ✅")

---
## 10. Тестуємо свій API всередині ноутбука 🧪

Щоб не запускати окремий сервер, FastAPI дає `TestClient` — він викликає застосунок **у пам'яті** (мережа не потрібна). Інтерфейс — як у `requests`.

In [3]:
from fastapi.testclient import TestClient

client = TestClient(app)

# GET кореня
print("GET /        ->", client.get("/").json())

# GET з path + query
print("GET /items/7 ->", client.get("/items/7", params={"q": "пошук"}).json())

# POST з коректним тілом
r = client.post("/items", json={"name": "Клавіатура", "price": 1200})
print("POST /items  ->", r.status_code, r.json())

GET /        -> {'message': 'Привіт, API!'}
GET /items/7 -> {'detail': 'Not Found'}
POST /items  -> 404 {'detail': 'Not Found'}


In [ ]:
# POST з НЕкоректним тілом: price = -5 (а має бути > 0)
r = client.post("/items", json={"name": "Миша", "price": -5})
print("Статус:", r.status_code, "(422 = Unprocessable Entity)")
print("FastAPI сам пояснив, що не так:")
import json as _json
print(_json.dumps(r.json(), ensure_ascii=False, indent=2))

---
## 11. Повний CRUD-сервіс: «нотатки» 📒

Зберемо все разом — повноцінний REST-ресурс із in-memory сховищем (словник). У реальному застосунку замість словника була б база даних.

| Метод | Шлях | Дія |
|---|---|---|
| `GET` | `/notes` | список усіх нотаток |
| `GET` | `/notes/{id}` | одна нотатка (або `404`) |
| `POST` | `/notes` | створити (`201`) |
| `PUT` | `/notes/{id}` | оновити (або `404`) |
| `DELETE` | `/notes/{id}` | видалити (`204`) |

In [ ]:
from fastapi import FastAPI, HTTPException

notes_app = FastAPI(title="Notes API")

class NoteIn(BaseModel):
    title: str = Field(min_length=1)
    text: str = ""

class Note(NoteIn):
    id: int

DB: dict[int, Note] = {}     # «база даних» у пам'яті
_next_id = 1

@notes_app.get("/notes")
def list_notes() -> list[Note]:
    return list(DB.values())

@notes_app.get("/notes/{note_id}")
def get_note(note_id: int) -> Note:
    if note_id not in DB:
        raise HTTPException(status_code=404, detail="Нотатку не знайдено")
    return DB[note_id]

@notes_app.post("/notes", status_code=201)
def create_note(payload: NoteIn) -> Note:
    global _next_id
    note = Note(id=_next_id, **payload.model_dump())
    DB[_next_id] = note
    _next_id += 1
    return note

@notes_app.put("/notes/{note_id}")
def update_note(note_id: int, payload: NoteIn) -> Note:
    if note_id not in DB:
        raise HTTPException(status_code=404, detail="Нотатку не знайдено")
    updated = Note(id=note_id, **payload.model_dump())
    DB[note_id] = updated
    return updated

@notes_app.delete("/notes/{note_id}", status_code=204)
def delete_note(note_id: int):
    if note_id not in DB:
        raise HTTPException(status_code=404, detail="Нотатку не знайдено")
    del DB[note_id]

print("Notes API готовий ✅")

In [ ]:
# Прокрутимо повний життєвий цикл нотатки через TestClient
nc = TestClient(notes_app)

print("Спочатку порожньо:", nc.get("/notes").json())

# CREATE
r = nc.post("/notes", json={"title": "Купити каву", "text": "арабіка"})
note = r.json()
print("POST   ->", r.status_code, note)
nid = note["id"]

# READ (один)
print("GET id ->", nc.get(f"/notes/{nid}").json())

# UPDATE
r = nc.put(f"/notes/{nid}", json={"title": "Купити чай", "text": "зелений"})
print("PUT    ->", r.status_code, r.json())

# READ (список)
print("GET all->", nc.get("/notes").json())

# DELETE
print("DELETE ->", nc.delete(f"/notes/{nid}").status_code, "(204)")
print("Після видалення GET id ->", nc.get(f"/notes/{nid}").status_code, "(404)")

---
## 12. Як запустити сервер по-справжньому 🚀

`TestClient` зручний для тестів, але живий сервер піднімають через **uvicorn**. Збережіть застосунок у файл, наприклад `main.py`:

```python
# main.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Привіт!"}
```

І запустіть із терміналу:

```bash
uvicorn main:app --reload
```

Сервер підніметься на `http://127.0.0.1:8000`. Найприємніше — FastAPI **автоматично** генерує інтерактивну документацію:

- **Swagger UI** → `http://127.0.0.1:8000/docs` (можна слати запити прямо з браузера)
- **ReDoc** → `http://127.0.0.1:8000/redoc`
- **OpenAPI-схема** (JSON-контракт усього API) → `http://127.0.0.1:8000/openapi.json`

> ⚠️ У ноутбуці `uvicorn.run(...)` заблокує клітинку (сервер працює нескінченно). Тому для навчання зручніший `TestClient`, а повноцінний запуск роблять окремим файлом/процесом.

Подивімось на згенеровану OpenAPI-схему нашого Notes API — це той самий «контракт», який ми вручну читали в чужих API:

In [ ]:
schema = notes_app.openapi()
print("Назва API:", schema["info"]["title"])
print("\nЕndpoint-и з OpenAPI-схеми:")
for path, methods in schema["paths"].items():
    print(" ", path, "->", [m.upper() for m in methods])

---
## Підсумок 📌

**Споживання API (ми — клієнт):**

| Інструмент | Призначення |
|---|---|
| `requests.get/post/put/patch/delete` | HTTP-запити |
| `params=` / `headers=` / `json=` | query-параметри / заголовки / JSON-тіло |
| `resp.status_code` / `resp.json()` | код відповіді / тіло як dict |
| `resp.raise_for_status()` | кинути виняток на `4xx`/`5xx` |
| `timeout=` | захист від «зависання» |
| `Authorization: Bearer <token>` | автентифікація (так само й до LLM) |
| `httpx` + `asyncio` | багато запитів паралельно |

**Створення API (ми — сервер):**

| Інструмент | Призначення |
|---|---|
| `FastAPI()` | застосунок |
| `@app.get/post/put/delete` | маршрути (CRUD) |
| `{item_id}` + анотація типу | path-параметр |
| аргумент Pydantic-моделі | тіло запиту + автоматична валідація |
| `HTTPException(404, ...)` | повернути помилку з потрібним кодом |
| `TestClient` | тестувати застосунок у пам'яті |
| `uvicorn main:app --reload` | запустити живий сервер |
| `/docs`, `/openapi.json` | автодокументація та контракт |

**Зв'язок з попередніми темами:** тіло запиту/відповіді — це **JSON**; його форму описує **Pydantic-модель** (валідація!); сам застосунок — це **класи й функції** (ООП). API просто з'єднує все це через мережу.

### 🎯 Завдання для самоперевірки

1. **Клієнт:** через `requests` отримайте список користувачів з `https://jsonplaceholder.typicode.com/users` і виведіть `name` та `email` кожного. Додайте обробку помилок із розділу 5.
2. **Сервер:** додайте до `notes_app` query-параметр `?q=` у `GET /notes`, щоб повертати лише нотатки, чий `title` містить підрядок `q`. Перевірте через `TestClient`.
3. **Статус-коди:** зробіть так, щоб `POST /notes` із порожнім `title` повертав `422`, і поясніть (у markdown), чому це відбувається автоматично.
4. **Інтеграція:** напишіть endpoint `GET /weather/{city}`, який *усередині* робить запит до зовнішнього погодного API (наприклад, `open-meteo.com`) і повертає спрощену відповідь. Так ваш API стане «обгорткою» над чужим — типовий патерн для tools у AI-агентах.

➡️ Далі: погляньте, як LLM-сервіси у вашому проєкті (`services/`) під капотом — це теж REST-виклики з токеном; а tools для агентів (`tools/`) часто обгортають зовнішні API саме так, як у завданні 4.
